# [실습] 오픈 모델 에이전트 (Ollama)

Ollama로 오픈 모델을 실행하고, LangGraph 에이전트에 연결합니다.

## 학습 목표

- `init_chat_model("ollama:...")` 또는 `ChatOllama`로 Ollama 모델을 LangChain에 연결
- 같은 `create_agent` 구성에서 `model` 파라미터만 바꿔 모델 교체
- 동일 태스크에서 모델별 도구 호출 흐름 비교

## 1. Ollama 설치 및 모델 다운로드

https://ollama.com/

Ollama는 오픈 모델을 로컬에서 실행하는 런타임입니다.
GPU가 없는 환경에서도 동작하며, 모델 다운로드, 서빙, 관리 명령을 제공합니다.

### Ollama 설치

- Linux (RunPod 등 GPU 서버)
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

- Windows: 홈페이지에서 설치 파일을 실행

Ollama는 기본적으로 11434 포트에서 실행됩니다.
사용 가능한 모델 목록은 https://ollama.com/models 에서 확인할 수 있습니다.

### 1-1. Ollama 설치 및 서빙 시작

### 설치, 서빙, 다운로드 셀 안내

아래 셀들은 처음 환경을 구성할 때만 실행합니다.
기본 상태에서는 주석 처리해 둡니다.

- 이미 Ollama가 설치되어 서빙 중이면 다음 절로 넘어갑니다.
- 처음 실행하는 경우 OS에 맞는 셀의 주석을 풀고 실행합니다.
- Windows에서는 Ollama 실행 여부를 확인하고, 필요할 때만 `ollama serve` 셀을 실행합니다.
- GPU가 없는 환경에서는 27B 모델 추론이 느립니다. 9B 등 소형 모델로 대체할 수 있습니다.

In [ ]:
# Ollama 설치 (이미 설치되어 있으면 건너뜁니다)
# !curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Ollama 서빙 시작 (백그라운드 실행)
# !ollama serve &

### 1-2. 모델 다운로드

`ollama pull` 명령으로 모델을 다운로드합니다.
- qwen3.6:35b: 메인 모델


In [ ]:
# !ollama pull qwen3.6:35b

In [ ]:
# 다운로드된 모델 확인
# !ollama list

In [ ]:
# GPU 상태 확인
# !nvidia-smi

---
## 2. Ollama 모델 연결

In [ ]:
%pip install beautifulsoup4 slack-sdk youtube-transcript-api langchain_community dotenv langchain-tavily  langchain langchain-openai langchain-ollama -q

### 2-1. init_chat_model 방식

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv('env', override=True)

from langchain.chat_models import init_chat_model

# provider:model 형식으로 Ollama 모델을 불러옵니다.
ollama_llm = init_chat_model("ollama:qwen3.6:35b")

print(f'모델 타입: {type(ollama_llm).__name__}')

### 2-2. ChatOllama 직접 사용

In [ ]:
from langchain_ollama import ChatOllama

ollama_direct = ChatOllama(model="qwen3.6:35b")

print(f'모델 타입: {type(ollama_direct).__name__}')
print(f'모델명: {ollama_direct.model}')

### 2-3. 기본 호출 테스트

In [ ]:
# Ollama 모델 기본 호출 테스트
response = ollama_llm.invoke('안녕하세요! 한 문장으로 자기 소개 부탁드립니다.')
print(f'Ollama 응답: {response.text}')
print(f'사용량: {response.usage_metadata}')

In [ ]:
# 스트리밍 테스트
for chunk in ollama_llm.stream('대한민국의 수도는 어디인가요?'):
    print(chunk.text, end='')

### Ollama 모델 상태 확인

```
ollama list    로컬 모델 목록
ollama ps      실행 중인 모델
```

---
## 3. 오픈 모델 에이전트 구동

`test_tools.py`의 도구를 활용하여 에이전트를 구성합니다.

### 3-1. 도구 로드

In [ ]:
from test_tools import (
    tavily_search, fetch_url, run_command, load_skill,
    write_slack_message, python_repl, read_file, write_file,
    youtube_transcript,
)

tools = [
    tavily_search, fetch_url, run_command, load_skill,
    write_slack_message, python_repl, read_file, write_file,
    youtube_transcript,
]

print(f'도구 {len(tools)}개 로드 완료: {[t.name for t in tools]}')

### 3-2. GPT-5.2 에이전트

In [ ]:
from langchain.agents import create_agent
import time

SYSTEM_PROMPT = "주어진 도구를 사용하여 질문에 답변하세요. 매 도구를 사용하기 전 중간 상황을 간단히 설명하세요."

gpt_agent = create_agent(
    model="gpt-5.2",
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

TASK = "Tavily로 'Multi Agent Context Engineering'을 검색하고, 가장 관련 높은 URL 하나를 골라 fetch한 뒤, 핵심 내용을 3줄로 요약해주세요."

start = time.time()
gpt_result = gpt_agent.invoke(
    {"messages": [{"role": "user", "content": TASK}]},
    config={"configurable": {"thread_id": "gpt-compare"}},
)
gpt_time = time.time() - start

print(f'[GPT-5.2] 소요 시간: {gpt_time:.2f}초')
print(f'응답: {gpt_result["messages"][-1].text}')

### 3-3. qwen3.6:35b 에이전트 (Ollama)

In [ ]:
ollama_agent = create_agent(
    model=ollama_llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

start = time.time()
ollama_result = ollama_agent.invoke(
    {"messages": [{"role": "user", "content": TASK}]},
    config={"configurable": {"thread_id": "ollama-compare"}},
)
ollama_time = time.time() - start

print(f'[qwen3.6:35b] 소요 시간: {ollama_time:.2f}초')
print(f'응답: {ollama_result["messages"][-1].text}')

---
## 4. 비교 분석

### 4-1. Tool Calling 패턴 비교

두 모델의 도구 호출 수, 도구 이름, 인자를 비교합니다.

In [ ]:
def analyze_tool_calls(result, model_name):
    """에이전트 결과에서 도구 이름과 인자를 추출합니다."""
    tool_calls = []
    total_messages = len(result["messages"])

    for msg in result["messages"]:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                tool_calls.append({
                    'name': tc['name'],
                    'args': tc['args'],
                })

    print(f'\n=== {model_name} ===')
    print(f'총 메시지 수: {total_messages}')
    print(f'도구 호출 수: {len(tool_calls)}')
    for i, tc in enumerate(tool_calls, 1):
        print(f'  {i}. {tc["name"]}({tc["args"]})')
    print(f'최종 응답: {result["messages"][-1].text[:200]}')
    return tool_calls

gpt_tools = analyze_tool_calls(gpt_result, "GPT-5.2")
ollama_tools = analyze_tool_calls(ollama_result, "qwen3.5:27b")

### 4-2. 실행 지표 요약

In [ ]:
print(f'{"지표":<20} {"GPT-5.2":>15} {"qwen3.5:27b":>15}')
print('-' * 52)
print(f'{"소요 시간":<20} {gpt_time:>14.2f}s {ollama_time:>14.2f}s')
print(f'{"도구 호출 수":<20} {len(gpt_tools):>15} {len(ollama_tools):>15}')
print(f'{"메시지 수":<20} {len(gpt_result["messages"]):>15} {len(ollama_result["messages"]):>15}')

# 검색 도구를 사용했는지 확인
gpt_names = [tc['name'] for tc in gpt_tools]
ollama_names = [tc['name'] for tc in ollama_tools]

gpt_searched = 'tavily_search' in gpt_names
ollama_searched = 'tavily_search' in ollama_names

print(f'{"검색 도구 사용":<20} {str(gpt_searched):>15} {str(ollama_searched):>15}')

### 4-3. 여러 태스크로 비교

In [ ]:
test_tasks = [
    "현재 디렉토리에 어떤 .py 파일이 있는지 확인해주세요.",
    "test_tools.py를 읽고, 정의된 함수 이름만 리스트로 출력하는 Python 코드를 작성해서 실행해주세요.",
    "유튜브 영상 https://www.youtube.com/watch?v=9vM4p9NN0Ts 의 자막을 가져와서 핵심 내용을 3줄로 요약해주세요.",
]

results = []
for i, task in enumerate(test_tasks):
    row = {"task": task[:30]}

    for model_name, agent_fn in [("GPT-5.2", gpt_agent), ("qwen3.5:27b", ollama_agent)]:
        try:
            start = time.time()
            r = agent_fn.invoke(
                {"messages": [{"role": "user", "content": task}]},
                config={"configurable": {"thread_id": f"bench-{model_name}-{i}"}},
            )
            elapsed = time.time() - start
            tc_count = sum(len(msg.tool_calls) for msg in r["messages"]
                         if hasattr(msg, 'tool_calls') and msg.tool_calls)
            row[f"{model_name}_time"] = f"{elapsed:.1f}s"
            row[f"{model_name}_tools"] = tc_count
        except Exception as e:
            row[f"{model_name}_time"] = "Error"
            row[f"{model_name}_tools"] = 0

    results.append(row)

# 결과 테이블 출력
print(f'{"태스크":<32} {"GPT시간":>8} {"GPT도구":>6} {"Ollama시간":>10} {"Ollama도구":>8}')
print('-' * 70)
for row in results:
    print(f'{row["task"]:<32} {row["GPT-5.2_time"]:>8} {row["GPT-5.2_tools"]:>6} '
          f'{row["qwen3.5:27b_time"]:>10} {row["qwen3.5:27b_tools"]:>8}')

---
## [실습] 다양한 오픈 모델로 에이전트 구동하기

Ollama에서 다른 모델을 다운로드하여 에이전트를 만들고, 도구 선택과 인자 추출을 비교하세요.

진행 방법:
1. https://ollama.com/models 에서 원하는 모델 선택
2. `!ollama pull <모델명>`으로 다운로드
3. `init_chat_model("ollama:<모델명>")`으로 LLM 생성
4. `create_agent`로 에이전트를 만들고 같은 태스크 실행

관찰 포인트:
- 도구 선택 정확도
- 파라미터 추출 품질
- 다단계 도구 호출 안정성

In [ ]:
# 모델 다운로드
# !ollama pull qwen3.5:9b

# 에이전트 생성 및 실행
# my_llm = init_chat_model("ollama:qwen3.5:9b")
# my_agent = create_agent(model=my_llm, tools=tools, system_prompt=SYSTEM_PROMPT)
# result = my_agent.invoke(
#     {"messages": [{"role": "user", "content": TASK}]},
#     config={"configurable": {"thread_id": "my-test"}},
# )
# print(result["messages"][-1].text)


## build_agent로 배포 에이전트 만들기

배포 팩토리 build_agent.py는 `platform='ollama'`와 model_name으로 Ollama 모델을 고릅니다. 같은 에이전트가 로컬 Ollama 런타임에서 동작합니다. 모델과 도구 구성은 build_agent가 맡으므로 Slack 봇은 build_agent()만 호출합니다.

In [ ]:
%%writefile build_agent.py
"""build_agent.py

에이전트를 한 곳에서 조립하는 팩토리입니다. 모델은 select_model로 고르고,
표준 도구(tools.py)에 MCP 서버 도구를 더해 에이전트를 만듭니다. 서비스 코드(예: Slack 봇)는
build_agent()만 호출해 같은 에이전트를 받습니다.

MCP 도구를 await로 받아오므로 build_agent는 async 함수입니다.
"""

from __future__ import annotations

import os

from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

from select_model import load_model
from tools import DEFAULT_TOOLS
from mcp_config import load_servers

DEFAULT_SYSTEM_PROMPT = (
    '당신은 유용한 AI 어시스턴트입니다. '
    '필요하면 도구를 사용하고, 도구를 실행하기 전 중간 과정을 간략히 설명하세요.'
)

# build_agent가 직접 연결하는 MCP 서버. mcp_servers.json의 stateless 서버를 읽는다.
# stateful 세션(Playwright 등)은 봇이 세션으로 열어 extra_tools로 넘긴다.
MCP_SERVERS = load_servers(stateful=False)


async def _auto_elicitation(mcp_context, params, context):
    """Slack MCP의 post_message가 채널을 되물을 때 기본 채널로 자동 응답합니다.

    봇이나 노트북에는 사용자에게 입력 폼을 표시할 방법이 없으므로, SLACK_CHANNEL_ID
    환경변수의 채널로 답합니다.
    """
    from mcp.types import ElicitResult

    channel = os.environ.get('SLACK_CHANNEL_ID', 'C_DRYRUN')
    return ElicitResult(action='accept', content={'channel_id': channel})


async def load_mcp_tools(servers=None):
    """MCP 서버에서 도구를 받아옵니다. 연결에 실패하면 빈 목록을 돌려줍니다."""
    servers = MCP_SERVERS if servers is None else servers
    if not servers:
        return []
    try:
        from langchain_mcp_adapters.callbacks import Callbacks

        client = MultiServerMCPClient(
            servers, callbacks=Callbacks(on_elicitation=_auto_elicitation)
        )
        return await client.get_tools()
    except Exception as e:
        print(f'[build_agent] MCP 도구 로드 실패, 표준 도구만 사용합니다: {e}')
        return []


async def build_agent(
    model_name=None,
    platform='openai',
    model=None,
    tools=None,
    extra_tools=None,
    system_prompt=None,
    checkpointer=None,
    mcp_servers=None,
    **model_kwargs,
):
    """현재까지 구성된 에이전트를 만들어 반환합니다.

    Args:
        model_name: 모델 이름. select_model.load_model로 전달됩니다.
        platform: 'openai' | 'vllm' | 'ollama'. load_model로 전달됩니다.
        model: 이미 만든 chat model. 주면 model_name/platform은 무시됩니다.
        tools: 표준 도구 대신 쓸 도구 목록. 생략 시 tools.py의 DEFAULT_TOOLS.
        extra_tools: 호출자가 직접 만든 도구를 추가로 붙입니다. 봇이 stateful
            세션(예: Playwright)에서 받은 도구를 넘길 때 씁니다.
        system_prompt: 시스템 프롬프트. 생략 시 기본값을 씁니다.
        checkpointer: 멀티턴 대화를 위한 체크포인터.
        mcp_servers: 연결할 MCP 서버 설정. 생략 시 MCP_SERVERS, {}면 MCP를 끕니다.
        model_kwargs: load_model로 전달되는 추가 인자 (reasoning_effort 등).
    """
    if model is None:
        model = load_model(model_name, platform=platform, **model_kwargs)
    base_tools = list(DEFAULT_TOOLS) if tools is None else list(tools)
    mcp_tools = await load_mcp_tools(mcp_servers)
    extra = list(extra_tools) if extra_tools else []
    if system_prompt is None:
        system_prompt = DEFAULT_SYSTEM_PROMPT
    return create_agent(
        model,
        tools=base_tools + mcp_tools + extra,
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )

build_agent.py를 import해 Ollama 플랫폼으로 에이전트를 만듭니다. `mcp_servers={}`로 tools.py의 표준 도구만 사용합니다.

In [ ]:
from build_agent import build_agent

agent = await build_agent(platform='ollama', model_name='qwen3.6:35b', mcp_servers={})

result = await agent.ainvoke({
    'messages': [{'role': 'user', 'content': '123 곱하기 47을 계산하고 결과를 한 문장으로 알려줘.'}]
})
print(result['messages'][-1].text)

---
## 정리

### 핵심 내용

- `init_chat_model("ollama:model")`로 Ollama 모델을 LangChain 에이전트에 연결합니다.
- `create_agent`의 `model` 파라미터만 바꾸면 도구와 프롬프트 구성을 유지한 채 모델을 교체할 수 있습니다.
- 같은 태스크에서도 모델별 도구 선택 순서, 파라미터 추출 형태, 다단계 호출 길이가 다르게 나타납니다.

### 오픈 모델과 상용 모델의 비교

| 관점 | 오픈 모델 (Ollama) | 상용 모델 (API) |
|------|-------------------|-----------------|
| 비용 | 로컬 실행 환경 비용 발생 | 토큰당 과금 |
| 데이터 경로 | 모델 입력은 로컬 Ollama 런타임에서 처리 | 모델 제공자에게 전송 |
| 운영 | 모델, 런타임을 직접 관리 | 제공자가 관리 |
| 도구 호출 형식 | 모델별 안정성 편차가 존재 | function calling이 일관되게 지원 |

### 선택 기준

- 도구 호출 정확도와 인자 추출이 핵심이면 실행 결과를 우선 비교합니다.
- 데이터 경로와 운영 비용이 중요하면 Ollama 실행 환경과 API 모델을 함께 검토합니다.